# Automated discovery of potential behavioral findings -- Figure 3 and result tables
Rebuilds the effect landscape across all currently tested hypotheses and the aggregate and per-evidence-class result tables.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'run_analysis': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from aerobat.analysis.paper import (
    behavioral_evidence_class_findings_table,
    behavioral_findings_table,
    behavioral_finding_numbers,
    plot_effect_landscape,
)
from aerobat.analysis.tables import write_evidence_class_results_latex, write_results_latex
OUTPUT = ROOT / 'run_analysis' / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
findings, numbers = behavioral_finding_numbers(ROOT / 'results' / 'GPT-5-mini')
display(behavioral_findings_table(findings, significant_only=True))
print({key: numbers[key] for key in ['n_hypotheses_tested', 'n_bf10_ge_3', 'spearman_bf10_vs_p_tau']})

In [ ]:
import math
from itertools import combinations
from pathlib import Path
from typing import Any, Mapping

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

BF_EFFECT_THRESHOLD = 3.0
P_TAU_THRESHOLD = 0.05
CONFIGURATION_COMPONENT_ORDER = ["objective", "authority", "constraints", "situational context", "resource", "actor", "risk & return"]
POS, NEG, MID = "#E34948", "#2A78D6", "#F0EFEC"
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0B0B0B", "#52514E", "#898781", "#E1E0D9", "#C3C2B7", "#FFFFFF"
EFFECT_COLORS = {"positive": POS, "negative": NEG, "no_effect": MUTED, "inconclusive": MUTED, "direction_unresolved": MUTED}
EFFECT_CLASS_LABELS = {
    "positive": "Positive",
    "negative": "Negative",
    "no_effect": "No effect",
    "inconclusive": "Inconclusive",
    "direction_unresolved": "Inconclusive",
}
DIVERGING = LinearSegmentedColormap.from_list("aerobat_div", [NEG, MID, POS])


def _marker(effect_class: str, size: float = 26) -> dict[str, Any]:
    marker = "^" if effect_class == "positive" else "v" if effect_class == "negative" else "o"
    filled = effect_class in {"positive", "negative", "no_effect"}
    color = EFFECT_COLORS.get(effect_class, MUTED)
    return {"marker": marker, "s": size, "facecolor": color if filled else SURFACE, "edgecolor": color, "linewidth": 0.9}


def save_figure(fig: plt.Figure, output_stem: str | Path) -> None:
    stem = Path(output_stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    for suffix in (".pdf", ".png"):
        fig.savefig(stem.with_suffix(suffix), dpi=300, bbox_inches="tight", facecolor="white")



def _plot_effect_landscape(frame: pd.DataFrame, output_stem: str | Path | None = None) -> plt.Figure:
    data = frame.copy()
    preferred_columns = [
        ["compete", "distrust", "plan", "strategic aggression", "non-compliance", "extroversion"],
        ["purchase", "deceive", "friendliness", "empathy", "literalism", "sycophancy"],
    ]
    present = set(data.behavior_name)
    columns = [[behavior for behavior in column if behavior in present] for column in preferred_columns]

    behavior_labels = {
        "strategic aggression": "strategic\naggression",
    }

    variable_labels = {
        "Relative performance emphasis": "Relative perf. emphasis",
        "Role interdependence structure": "Role interdependence",
        "Communication overconfidence": "Actor overconfidence",
        "Opponent aggressiveness level": "Opponent aggression level",
        "Access to counter-attitudinal data": "Access to attitudinal data",
        "Constraint-instruction conflict": "Instruction conflict",
        "Constraint_complexity_and_conflict": "Constraint complexity & conflict",
        "Cost-minimization goal strength": "Cost-minimization goal",
        "Counterparty_interaction_style": "Actor friendliness",
        "Counterparty_vulnerability_cues": "Vulnerability cues",
        # "Emotional expression constraints": "Emotional constraints",
        "Intervention authority over others": "Intervention authority",
        "Normative_deception_modeling_by_actors": "Deception norms",
        "Peer purchasing descriptive norms": "Peer purchase norms",
        # "Relationship continuity expectation": "Continuity expectation",
        "Reward for throughput efficiency": "Reward for efficiency",
        "Short_term_payoff_weighting": "Short-term payoff",
        "Uncertainty of sanctions for aggression": "Sanction uncertainty",
        "Unfriendliness_penalty_severity": "Unfriendly penalty",
        "Vendor persuasiveness intensity": "Vendor persuasion",
    }

    def format_variable(value: Any) -> str:
        label = variable_labels.get(str(value), str(value).replace("_", " "))
        return " ".join(label.split())

    def layout(behaviors: list[str]) -> tuple[list[tuple[Any, float]], list[tuple[str, float, float]], float]:
        rows, spans, cursor = [], [], 0.0
        for behavior in behaviors:
            block = data[data.behavior_name == behavior].sort_values("Delta", ascending=False)
            if block.empty:
                continue
            start = cursor
            for row in block.itertuples(index=False):
                rows.append((row, cursor))
                cursor += 1.0
            spans.append((behavior, start - 0.45, cursor - 0.55))
            cursor += 1.05
        return rows, spans, max(cursor - 1.05, 0.0)

    layouts = [layout(column) for column in columns]
    max_y = max(item[2] for item in layouts)
    fig = plt.figure(figsize=(9.18, 8.13), facecolor=SURFACE)
    outer = fig.add_gridspec(1, 2, wspace=0.10, left=0.018, right=0.992, top=0.905, bottom=0.065)

    for column_index, (rows, spans, _) in enumerate(layouts):
        inner = outer[0, column_index].subgridspec(
            1, 4, width_ratios=[0.12, 0.43, 0.225, 0.225], wspace=0.055
        )
        ax_behavior = fig.add_subplot(inner[0, 0])
        ax_label = fig.add_subplot(inner[0, 1], sharey=ax_behavior)
        ax_delta = fig.add_subplot(inner[0, 2], sharey=ax_behavior)
        ax_bf = fig.add_subplot(inner[0, 3], sharey=ax_behavior)

        for ax in (ax_behavior, ax_label, ax_delta, ax_bf):
            ax.set_ylim(max_y - 0.35, -0.85)
            ax.set_yticks([])

        ax_behavior.set_xlim(0, 1)
        ax_label.set_xlim(0, 1)
        ax_behavior.axis("off")
        ax_label.axis("off")

        for row, y in rows:
            effect = row.effect_class
            color = EFFECT_COLORS.get(effect, MUTED)
            weight = "bold" if effect in {"positive", "negative"} else "normal"
            label_color = INK if effect in {"positive", "negative"} else INK2
            ax_label.text(0.85, y, format_variable(row.variable), va="center", ha="right", fontsize=7.4, color=label_color, fontweight=weight)
            ax_delta.plot([row.Delta_low, row.Delta_high], [y, y], color=color, alpha=0.55, lw=1.55, solid_capstyle="round")
            ax_delta.scatter([row.Delta], [y], zorder=3, **_marker(effect, 24))
            ax_bf.scatter([row.log10_bf10], [y], zorder=3, **_marker(effect, 19))

        for behavior, start, end in spans:
            ax_behavior.plot([0.46, 0.46], [start, end], color=AXIS, lw=1.35, clip_on=False)
            ax_behavior.text(
                0.16,
                (start + end) / 2,
                behavior_labels.get(behavior, behavior),
                rotation=90,
                va="center",
                ha="center",
                fontsize=8.4,
                color=INK,
                fontweight="bold",
                linespacing=0.9,
                clip_on=False,
            )

        ax_delta.set_xlim(-3.45, 6.05)
        ax_delta.set_xticks([0, 5])
        ax_bf.set_xlim(-1.45, 15.95)
        ax_bf.set_xticks([0, 5, 10, 15])
        for ax, label in ((ax_delta, r"$\Delta$ (95% CrI)"), (ax_bf, r"$\log_{10}\mathrm{BF}_{10}$")):
            ax.axvline(0, color=AXIS, lw=1.05, zorder=0)
            ax.grid(axis="x", color=GRID, lw=0.65, zorder=0)
            ax.tick_params(axis="x", which="both", top=True, labeltop=True, bottom=True, labelbottom=True, length=0, pad=2.5, colors=INK2, labelsize=8.8)
            ax.tick_params(axis="y", length=0)
            ax.text(0.5, 1.025, label, transform=ax.transAxes, ha="center", va="bottom", fontsize=9.4, color=INK)
            ax.set_xlabel(label, fontsize=9.4, color=INK, labelpad=4)
            for side in ("left", "right", "top"):
                ax.spines[side].set_visible(False)
            ax.spines["bottom"].set_color(AXIS)
            ax.spines["bottom"].set_linewidth(0.75)

        ax_behavior.text(0.16, 1.025, r"$Y$", transform=ax_behavior.transAxes, ha="center", va="bottom", fontsize=9.4, color=INK)
        ax_behavior.text(0.16, -0.035, r"$Y$", transform=ax_behavior.transAxes, ha="center", va="top", fontsize=9.4, color=INK)
        ax_label.text(0.5, 1.025, r"$X$", transform=ax_label.transAxes, ha="center", va="bottom", fontsize=9.4, color=INK)
        ax_label.text(0.5, -0.035, r"$X$", transform=ax_label.transAxes, ha="center", va="top", fontsize=9.4, color=INK)

    handles = [
        Line2D([], [], linestyle="none", marker="^", markersize=5.4, markerfacecolor=POS, markeredgecolor=POS, label="positive"),
        Line2D([], [], linestyle="none", marker="v", markersize=5.4, markerfacecolor=NEG, markeredgecolor=NEG, label="negative"),
        Line2D([], [], linestyle="none", marker="o", markersize=5.0, markerfacecolor=MUTED, markeredgecolor=MUTED, label="no effect"),
        Line2D([], [], linestyle="none", marker="o", markersize=5.0, markerfacecolor=SURFACE, markeredgecolor=MUTED, label="inconclusive"),
    ]
    legend = fig.legend(
        handles=handles,
        loc="upper center",
        ncol=4,
        frameon=True,
        bbox_to_anchor=(0.5, 0.995),
        fontsize=8.8,
        handletextpad=0.45,
        columnspacing=1.25,
        borderpad=0.35,
    )
    legend.get_frame().set_edgecolor(AXIS)
    legend.get_frame().set_linewidth(0.9)
    legend.get_frame().set_facecolor(SURFACE)
    # if output_stem:
    #     save_figure(fig, output_stem)
    return fig

In [ ]:
fig = plot_effect_landscape(findings, OUTPUT / 'fig3_effect_landscape')

In [ ]:
write_results_latex(behavioral_findings_table(findings, significant_only=True), OUTPUT / 'tabular_result_main.tex')
write_results_latex(behavioral_findings_table(findings, significant_only=False), OUTPUT / 'tabular_result_full.tex')
write_evidence_class_results_latex(behavioral_evidence_class_findings_table(findings), OUTPUT / 'tabular_result_table.tex')
